In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [30]:
df = pd.read_csv('quote_dataset.csv')

In [31]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [32]:
df.shape

(3038, 2)

In [33]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [34]:
quotes = df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: str

**<h3>Data Preprocessing</h3>**

*Making the text lower case and removing commas and puntuations*

In [35]:
quotes = quotes.str.lower()

In [36]:
import string
translator = str.maketrans('','',string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [37]:
quotes.head()

0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: str

**Tokenization**

In [38]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [39]:
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [40]:
word_index = tokenizer.word_index
print(len(word_index))

8978


In [41]:
sequence = tokenizer.texts_to_sequences(quotes)

In [42]:
for i in range(3):
    print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [43]:
for i in range(3):
    print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


**Input and output variable**

In [46]:
X = []
y = []

for seq in sequence:
    for i in range(1, len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [48]:
len(X), len(y)

(85271, 85271)

**Padding**

In [49]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [50]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [51]:
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [54]:
y = np.array(y)

In [56]:
X_padded.shape, y.shape

((85271, 745), (85271,))

**One hot encoding**

In [61]:
# Keep labels as integer class IDs to avoid allocating a dense one-hot matrix.
y_one_hot = np.asarray(y, dtype=np.int32)
print(y_one_hot.shape, y_one_hot.min(), y_one_hot.max())

# Compile the model with loss="sparse_categorical_crossentropy".

(85271,) 1 8978


**Basic RNN model**

In [62]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SimpleRNN

In [63]:
embedding_dim = 50
rnn_units = 128

In [64]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)

rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

c:\Users\MSI\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [66]:
rnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])